# 데모 4 — OpenTelemetry: 계측은 한 번, 백엔드는 자유

**한 줄 요약**: 데모 3에서는 Langfuse **SDK**(CallbackHandler)로 계측했다.
이번엔 같은 그래프를 **벤더 중립 표준(OTel + OpenInference)** 으로 계측한다.
트레이스를 받을 백엔드는 코드가 아니라 **환경변수만** 결정한다.

```mermaid
flowchart LR
    A["Adaptive RAG 그래프<br/>(코드에 벤더 SDK 없음)"] --> I["OpenInference<br/>LangChainInstrumentor"]
    I --> P["OTel TracerProvider<br/>SpanProcessor × N"]
    P -- "OTLP HTTP" --> L[("Langfuse<br/>:3000/api/public/otel")]
    P -- "OTLP HTTP" --> X[("Phoenix<br/>:6006/v1/traces")]
    P -.-> C["console (디버그)"]
    style P fill:#d0ebff
```

**before / after**
- before(데모 3): `from langfuse.langchain import CallbackHandler` — 벤더 SDK가 코드에 박힘.
  도구를 바꾸면 코드를 걷어내야 한다.
- after(지금): 코드에는 OTel 표준만. Langfuse ↔ Phoenix ↔ 사내 컬렉터 전환 = **env 수정**.
  심지어 **동시에 여러 백엔드로 fan-out**도 SpanProcessor 추가로 끝난다.

| 실행 방법 | OTEL_TARGETS | 결과 |
|---|---|---|
| `make demo4-langfuse` | `langfuse` | Langfuse 로 전송 (Basic Auth 헤더 자동 구성) |
| `make demo4-phoenix` | `phoenix` | Phoenix 로 전송 |
| `make demo4-both` | `langfuse,phoenix` | **두 곳 동시** (하이라이트 컷) |
| (표준 방식) | 빈 값 | `OTEL_EXPORTER_OTLP_ENDPOINT/_HEADERS` env 그대로 사용 |
| 오프라인 리허설 | `console` | 콘솔로 span 출력 (서버 불필요) |

⚠ Langfuse 의 OTLP 는 **HTTP(JSON/protobuf) 전용**이다 — gRPC exporter 를 쓰면 안 된다.
(그래서 이 데모는 `opentelemetry-exporter-otlp-proto-http` 를 사용한다)

In [ ]:
# ── 준비: 백엔드는 env 로만 결정된다 ─────────────────────────
import os
from common.console import banner, step, show_answer, stream_run, console
from common.llm import provider
from common.tracing import init_otel     # ← Langfuse "SDK" 는 어디에도 임포트하지 않는다!

TARGETS = os.getenv("OTEL_TARGETS", "console")   # 라이브에서는 make 타깃이 이 값을 바꾼다
banner("데모 4 — OTel 백엔드 스왑", f"LLM_PROVIDER = {provider()} / OTEL_TARGETS = {TARGETS}")

tracer_provider = init_otel(TARGETS)

위 셀이 하는 일의 전부:

1. `TracerProvider` 생성 + 대상별 `BatchSpanProcessor(OTLPSpanExporter)` 등록
   (`langfuse` 대상이면 `/api/public/otel/v1/traces` + Basic Auth 헤더를 자동 구성)
2. `LangChainInstrumentor().instrument(...)` — LangChain/LangGraph 실행 전체가
   OpenInference 시맨틱 컨벤션의 OTel span 이 된다.

이제 **데모 1의 그래프를 코드 수정 없이** 실행한다. callbacks 주입조차 없다.

In [ ]:
from demo1_workflow_to_loop.adaptive_rag import build_adaptive_rag

app = build_adaptive_rag()
for q in [
    "LangGraph에서 조건부 엣지는 무엇이고 언제 쓰나요?",
    "LangGraph로 human approval 단계를 넣으려면 어떻게 해요?",   # 루프 도는 트레이스
]:
    step(f"실행 — {q[:34]}…")
    final, _, path = stream_run(app, {"question": q, "rewrite_count": 0})
    console.print(f"  경로: [bold]{' → '.join(path)}[/]")
    show_answer(q, final["answer"])

In [ ]:
# ── 전송 보장(배치 플러시) 후 확인처 안내 ─────────────────────
tracer_provider.force_flush()
step("트레이스 확인처")
base = os.getenv("LANGFUSE_BASE_URL") or "http://localhost:3000"
if "langfuse" in TARGETS:
    print(f"  Langfuse: {base} → Traces (route_question/retrieve/... span 트리)")
if "phoenix" in TARGETS:
    print("  Phoenix : http://localhost:6006 → 프로젝트 default 의 Traces")
if "console" in TARGETS:
    print("  콘솔 출력에서 위 span JSON 을 직접 확인 (오프라인 리허설 모드)")
print("\n같은 실행, 같은 코드 — 받는 쪽만 바뀐다. 계측은 한 번, 백엔드는 자유.")

## 정리 — 발표 포인트

1. **이 노트북에는 `import langfuse` 가 없다.** OpenInference 계측 + OTLP exporter 뿐.
   Langfuse 는 그저 "OTLP 를 받는 백엔드 중 하나"로 강등(?)되었다.
2. **fan-out**: `OTEL_TARGETS=langfuse,phoenix` — SpanProcessor 를 2개 등록하면
   같은 실행이 두 대시보드에 동시에 나타난다. 도구 마이그레이션 기간에 특히 유용.
3. **주의 두 가지**: Langfuse OTLP 는 HTTP 전용(gRPC 미지원) / 세션·태그 같은
   trace-level 속성은 SDK 헬퍼가 없으므로 `langfuse.trace.*` span 속성 규약으로 넣어야 한다.
   (풍부한 속성이 필요하면 데모 3 방식, 표준·이식성이 우선이면 데모 4 방식)

**다음 데모 예고**: 지금까지는 "그래프 하나"를 다뤘다. 마지막으로 플래닝·서브에이전트·
파일시스템을 갖춘 **딥 에이전트**가 데모 1의 그래프를 부하 직원처럼 부리는 것을 보자 → 데모 5